Python is dynamically typed, so it does not through compile time error, when mismatched input is passed.
type hinting helps to address this by showing editor errors. But application can still compile and run. It will only throw error at runtime, if mismatch execution is there.

In [ ]:
# Types are hints — ignored at runtime
def summarize_test(text: str, max_tokens: int) -> str: 
    print(f"text:{text}, max_tokens: {max_tokens}")
    return text

def summarize(text: str, max_tokens: int) -> str: 
    response= text[:max_tokens] + "..." if len(text) > max_tokens else text
    print(response)
    return response


def extract_keywords(text: str) -> list[str]: ...

def score_responses(    responses: list[str]) -> dict[str, float]:
    return {response: len(response) for response in responses}

summarize_test(100,10) #typecheck fails but can run, as no runtime error
summarize(100,10) #typecheck fails: positional argument follows keyword argument

with python 3.9 the types like list,dict,tuple,set etc are built in.  
In previous version we need to import from typing package explictly.

In [ ]:
# Python 3.9+ — use built-in generics directly (no import needed)
def process_chunks(
    chunks: list[str],
    scores: dict[str, float],
    ids: tuple[str, ...],
    unique_models: set[str],
    result: str | None = None  # union with | (3.10+)
) -> list[dict[str, str | float]]:  # union with | (3.10+)
    ...

process_chunks(
    chunks=["chunk1", "chunk2"],
    scores={"chunk1": 0.5, "chunk2": 0.8}, 
    ids=("id1", "id2"), 
    unique_models={"model1", "model2"}, 
    result=None)

# Python 3.8 and earlier — import from typing
from typing import List, Dict, Tuple, Set, Union, Optional
def process_chunks_old(
    chunks: List[str],                      # same as list[str] in 3.9+
    scores: Dict[str, float],
    ids: Tuple[str,...],
    unique_models:Set[str],
    result: Optional[str] = None,           # same as str | None
) -> List[Dict[str, Union[str, float]]]:    # same as list[dict[str, str | float]]
    ...
    

**TypedDict** is used to describe the expected shape of a dictionary: which keys must exist, and what type each value should have.

**Literal** define allowed values for any field.

In [ ]:
from typing import TypedDict, Literal

class Message(TypedDict): #Message is a dictionary with 2 allowed keys, "role","content"
    role: Literal["user", "assistant", "system"]  # only these string values
    content: str



# Now this is type-checked:
messages: list[Message] = [
    {"role": "user", "content": "What is RAG?"},
]

# mypy/Pyright will catch this:
bad_messages: list[Message] = [
    {"role": "admin", "content": "..."},  # ERROR: "admin" not in Literal
]

**TypeAlias** gives a meaningful name to a complex type so annotations are easier to read and reuse.

**NamedTuple** creates a lightweight tuple-like structure with named fields. It keeps tuple behavior but allows attribute access like an object.

In [ ]:
from typing import TypeAlias, NamedTuple

EmbeddingVector: TypeAlias = list[float]
ChatMessage: TypeAlias = dict[str, str]

def average_score(scores: EmbeddingVector) -> float:
    return sum(scores) / len(scores)

vector: EmbeddingVector = [0.12, 0.45, 0.88]
message: ChatMessage = {"role": "user", "content": "Explain RAG"}

print(average_score(vector))
print(message["role"])

class UserProfile(NamedTuple):
    id: int
    name: str
    active: bool = True

user = UserProfile(1, "Alex")
print(user.name)   # attribute access
print(user[0])     # tuple-style access
print(user.active)

**NamedTuple** is best when you want an immutable, lightweight record that still behaves like a tuple.

Benefits of **NamedTuple**: memory-efficient, immutable, supports tuple unpacking, and works well for small fixed records.
Use **NamedTuple** when the data should not change after creation and tuple-style behavior is useful.

**dataclass** is better when you want a regular class with readable field definitions, optional mutability, and methods.
Use **dataclass** when you need to update values, add behavior, or model application objects more clearly than a tuple.

In [6]:
from dataclasses import dataclass
from typing import NamedTuple

class PointTuple(NamedTuple):
    x: int
    y: int

@dataclass
class PointData:
    x: int
    y: int

    def move(self, dx: int, dy: int) -> None:
        self.x += dx
        self.y += dy

point_tuple = PointTuple(10, 20)
point_data = PointData(10, 20)

print(point_tuple.x)
print(point_tuple[0])   # tuple-style access still works

point_data.move(5, 3)
print(point_data.x, point_data.y)

# point_tuple.x = 99   # error: NamedTuple is immutable
point_data.x = 99      # allowed: dataclass is mutable by default, can be made immutable by adding frozen=true
print(point_data)

p,q= point_tuple # unpack the values
print(p)
print(q)

10
10
15 23
PointData(x=99, y=23)
10
20


@overload decorator allow editor to understand overload versions. These are only used for type checking by editors or libraries like mypy

@overload only define method signetures , no implementation. these are not executed. they are used only by editors

In [ ]:

from typing import overload


@overload
def fetch_user(user_id: int) -> dict: ...
@overload
def fetch_user(user_id: list[int]) -> list[dict]: ...

def fetch_user(user_id: int | list[int]) -> dict | list[dict]:
    if isinstance(user_id, int):
        return {"id": user_id, "name": "Alice"}
    return [{"id": uid, "name": f"User-{uid}"} for uid in user_id]

user1 = fetch_user(1)          # returns dict
users = fetch_user([1, 2, 3])  # returns list[dict]